# Conservative 2D regrid — regions (grid → arbitrary polygons)

Gridded data → arbitrary polygon regions (countries, watersheds, ocean
basins, protected areas) is a canonical xagg-style workflow. Because
regions aren't a grid at all, `.conservative` can't express it;
`ConservativeRegridder.from_polygons` takes a numpy array of shapely
polygons and produces one conservatively-averaged value per region.

This notebook uses hand-built synthetic regions so it runs with zero
external downloads — swap in `geopandas.read_file(...)` for a real
shapefile and the rest is identical.

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.collections import PolyCollection
import shapely

import xarray_regrid  # noqa: F401
from xarray_regrid import ConservativeRegridder, polygons_from_coords

## Source — structured lat/lon field

A smooth analytic field with recognizable geographic pattern: a warm
band following the equator modulated by an east/west tilt. Gives visibly
different regional means depending on region location.

In [ ]:
lat = np.linspace(-80, 80, 160) + 0.5
lon = np.linspace(-180, 180, 360, endpoint=False) + 0.5
Lo, La = np.meshgrid(lon, lat)
field = np.cos(np.deg2rad(La)) ** 2 + 0.4 * np.sin(np.deg2rad(Lo))
src = xr.DataArray(
    field,
    dims=("latitude", "longitude"),
    coords={"latitude": lat, "longitude": lon},
)

## Regions — five hand-built polygons of varied shape

Box, circle, rotated rectangle, L-shape, and a polygon with a hole —
exercising the range of geometry the regridder accepts. These overlap
in places, which is fine: each region gets its own independent
area-weighted mean.

In [ ]:
def rotated_rect(cx, cy, w, h, angle_deg):
    theta = np.deg2rad(angle_deg)
    c = np.array([[-w/2, -h/2], [w/2, -h/2], [w/2, h/2], [-w/2, h/2]])
    R = np.array([[np.cos(theta), -np.sin(theta)],
                  [np.sin(theta),  np.cos(theta)]])
    return shapely.Polygon(c @ R.T + np.array([cx, cy]))

def circle(cx, cy, r, n=48):
    t = np.linspace(0, 2 * np.pi, n, endpoint=False)
    return shapely.Polygon(np.column_stack([cx + r*np.cos(t), cy + r*np.sin(t)]))

region_names = [
    "equatorial band",
    "northern island",
    "rotated block",
    "L-shape",
    "ring",
]
region_polys = np.array([
    shapely.box(-180, -15, 180, 15),
    circle(cx=-60, cy=50, r=18),
    rotated_rect(cx=80, cy=40, w=50, h=25, angle_deg=30),
    shapely.unary_union([
        shapely.box(-140, -60, -100, -20),
        shapely.box(-140, -60, -60, -50),
    ]),
    shapely.Polygon(
        shell=[(140, -40), (170, -40), (170, -5), (140, -5)],
        holes=[[(148, -30), (162, -30), (162, -12), (148, -12)]],
    ),
], dtype=object)
print(f"{len(region_polys)} regions, areas (deg²): "
      f"{[f'{shapely.area(p):.0f}' for p in region_polys]}")

## Build the regridder and apply

`from_polygons` takes source + target polygon arrays. We build the
source polygons from the grid's 1D coords via the `polygons_from_coords`
helper; the data gets flattened to match.

In [ ]:
src_polys = polygons_from_coords(lon, lat)

rgr = ConservativeRegridder.from_polygons(
    source_polygons=src_polys,
    target_polygons=region_polys,
    source_dim="src_cell",
    target_dim="region",
    target_coords=xr.Dataset(coords={"region": region_names}),
)
src_flat = xr.DataArray(src.values.ravel(), dims=("src_cell",))
regional = rgr.regrid(src_flat)
regional

## Source field + region outlines + regional means

In [ ]:
fig, (ax_map, ax_bar) = plt.subplots(
    1, 2, figsize=(13, 4.5), gridspec_kw={"width_ratios": [1.6, 1]},
)
src.plot(ax=ax_map, cmap="viridis", add_colorbar=True,
         cbar_kwargs={"shrink": 0.7, "label": "source field"})
patches = [np.asarray(p.exterior.coords) for p in region_polys]
pc = PolyCollection(
    patches, facecolor="none", edgecolor="white", lw=1.4,
)
ax_map.add_collection(pc)
for name, poly in zip(region_names, region_polys):
    c = poly.representative_point()
    ax_map.annotate(name, (c.x, c.y), color="white", fontsize=8,
                    ha="center", va="center")
ax_map.set_title("source + region outlines")
ax_map.set_xlim(-180, 180); ax_map.set_ylim(-80, 80)

ax_bar.barh(region_names, regional.values, color="tab:blue")
ax_bar.set_xlabel("area-weighted regional mean")
ax_bar.invert_yaxis()
ax_bar.grid(axis="x", alpha=0.3)
plt.tight_layout()

## Conservation check

Area-weighted sum of regional means × region areas should match the
direct A·s computation from the regridder's internal area matrix.

In [ ]:
A = rgr._areas                                # sparse (n_regions, n_src)
tgt_area = np.ravel(A.sum(axis=1).todense())
src_cover = np.ravel(A.sum(axis=0).todense())

direct = float((src.values.ravel() * src_cover).sum())
via_regrid = float((regional.values * tgt_area).sum())
print(f"direct   A·s            : {direct:.6f}")
print(f"Σ regional_mean · a_dst : {via_regrid:.6f}")
print(f"relative error          : {abs(direct - via_regrid) / abs(direct):.2e}")

## Reuse: persist the regridder to netCDF

The weight matrix is reusable. For any workflow that repeatedly
aggregates new source data onto the same region set, save once,
reload on subsequent runs to skip the intersection build.

In [ ]:
import tempfile
from pathlib import Path
path = Path(tempfile.gettempdir()) / "regions_regridder.nc"
rgr.to_netcdf(path)
print(f"wrote {path}  ({path.stat().st_size / 1024:.1f} KB)")

rgr2 = ConservativeRegridder.from_netcdf(path)
# Apply to a different source field, same grid:
other = xr.DataArray(
    (np.sin(np.deg2rad(Lo)) ** 2).ravel(), dims=("src_cell",),
)
print("regional means on a different field:")
for n, v in zip(region_names, rgr2.regrid(other).values):
    print(f"  {n:20s}: {v:+.4f}")